### Análise Preditiva Aplicada a Manutenção de Equipamentos Industriais

Estruturação de um Pipeline Preditivo aplicado ao contexto de Indústria 4.0.

Um parque fabril monitorado por sensores necessita prever quebras mecânicas nos equipamentos para evitar paradas na linha de produção.


In [ ]:
# importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

### Fase 1 - Análise Exploratória (EDA)

- Carregamento da base de dados
- Apresentação das dimensões do dataset, os tipos de dados das variáveis e o resumo estatístico descritivo das colunas numéricas.

In [ ]:
df = pd.read_csv("../data/manutencao_preditiva.csv", sep = ",")

In [ ]:
print("\n========== INSPEÇÃO INICIAL DO DATASET ===========")
print(f"-----> O Dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas")
print("\n\n-----> COLUNAS, TIPOS DE DADOS E DADOS AUSENTES:")
display(df.info())
print("\n-----> AMOSTRA DO DATASET:")
display(df.sample(10))
print("\n-----> ESTATÍSTICAS DESCRITIVAS:\n")
display(df.describe())

- Construção de gráficos para exploração dos dados

In [ ]:
'''
Construção de uma matriz de gráficos tipo histograma para exibição das
distribuições de probabilidades das variáveis quantitativas contínuas
'''

cols_num_continuas = ['temperatura_ar_k','temperatura_processo_k', 'velocidade_rotacao_rpm', 'torque_nm','desgaste_ferramenta_min']

# Configuração do estilo geral dos gráficos
sns.set_theme(style="white")
plt.rcParams['font.family'] = 'sans-serif'

# Instanciamento de dicionário associando nomes das colunas a rótulos informativos
lista_rotulos = ['Temperatura do Ar (K)', 'Temperatura do Processo (K)', 'Velocidade de Rotação (RPM)',
                'Torque (Nm)', 'Desgaste da Ferramenta (min)']
rotulos = dict(zip(cols_num_continuas, lista_rotulos))

# Criando a figura geral com os 5 subplots/gráficos
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 9))
axs_flat = axs.flat # adota numeração sequencial para referência aos subgráficos

# Definição de cores para os elementos dos gráficos
cor_base = 'slategray'
cor_texto = 'dimgrey'
cor_titulo = 'darkslategray' 

# Loop para plotagem de cada uma das variáveis
for i, v in enumerate(cols_num_continuas):
    ax = axs_flat[i]
    
    sns.histplot(df, x=v, ax=ax, color=cor_base, edgecolor="white", linewidth=0.5, alpha=0.85)
    
    ax.set_title(rotulos[v], fontsize=12, fontweight='bold', pad=12, color=cor_texto)
    ax.set_xlabel('') # Remove o rótulo do eixo X pois o título já descreve a variável
    ax.set_ylabel('Frequência (Registros)', fontsize=10, color=cor_texto)
    
    # Ajuste fino dos ticks (números dos eixos)
    ax.tick_params(axis='both', colors=cor_texto, labelsize=10)
    
    # Limpeza da poluição do gráfico, removendo as bordas e traços desnecessários
    sns.despine(ax=ax)

# Esconde o espaço do 6o gráfico para não ficar vazio (somente 5 variáveis / gráficos plotados)
axs_flat[-1].set_visible(False)

plt.suptitle("Distribuição das Variáveis Físicas dos Equipamentos", 
             fontsize=18, fontweight='bold', color=cor_titulo, x=0.05, y=0.98, ha='left')

# Ajuste fino do espaçamento dos subplots
plt.subplots_adjust(hspace=0.4, wspace=0.3, top=0.84, bottom=0.08, left=0.05, right=0.95)
plt.show()


A maioria dos sensores segue distribuições normais bem definidas, exceto o desgaste da ferramenta que possui comportamento uniforme.

In [ ]:
'''
Construção de um gráfico de barras empilhadas mostrando a distribuição da variável categórica "tipo"
já estratificadas conforme a variável alvo "falha_maquina", destacando a frequência de ocorreência de falhas.
'''

# Configuração do estilo e paleta de cores
sns.set_theme(style="white")
plt.rcParams['font.family'] = 'sans-serif'

cor_normal = 'slategrey'   # Cor para operação normal
cor_falha = 'crimson'      # Cor de alerta (vermelho) para a falha
cor_texto = 'dimgrey'      # Cor para textos gerais
cor_titulo = 'darkslategray' # Cor para o título principal

# Criação da figura do gráfico
fig, ax = plt.subplots(figsize=(10, 6))

# Cálculo da contagem de cada grupo e falha
# método size() anota as ocorrências dentro do groupby e unstack() "desempacota" num dataframe simples.
df_counts = df.groupby(['tipo', 'falha_maquina']).size().unstack(fill_value=0).reindex(['L', 'M', 'H'])

# Plota as barras com as ocorrências de máquinas sem falhas (normais)
ax.bar(df_counts.index, df_counts[0], label='Normal', color=cor_normal, alpha=0.85, width=0.6)

# Plota as ocorrências de falhas no topo (empilhadas)
ax.bar(df_counts.index, df_counts[1], bottom=df_counts[0], label='Falha', color=cor_falha, width=0.6)

#  Limpeza de elementos excessivos do gráfico
sns.despine(ax=ax)

ax.set_ylabel('Total de Equipamentos', fontsize=11, color=cor_texto)
ax.set_xlabel('Tipo de Equipamento', fontsize=11, color=cor_texto)
ax.tick_params(axis='both', colors=cor_texto, labelsize=11)

# Substitui as letras das categorias por rótulos informativos
ax.set_xticks(range(len(df_counts.index)))
ax.set_xticklabels(['Low (Baixo)', 'Medium (Médio)', 'High (Alto)'])

# Cálculo de porcentagens de falhas para adicionar rótulos aos gráficos
total_por_tipo = df_counts[0] + df_counts[1]
taxas_falha = (df_counts[1] / total_por_tipo) * 100

# laço para inserção de rótulos informativos dentro do gráfico
for i, tipo in enumerate(df_counts.index):
    # Total no topo de cada barra
    ax.text(i, total_por_tipo[tipo] + 100, f"{total_por_tipo[tipo]} un.", 
            ha='center', va='bottom', color=cor_texto, fontsize=10, fontweight='bold')
    
    # Texto destacando a taxa de falha interna
    ax.text(i, total_por_tipo[tipo] / 2, f"Falhas: {taxas_falha[tipo]:.1f}%", 
            ha='center', va='center', color='white', fontsize=11, fontweight='bold',
            bbox=dict(facecolor=cor_falha, alpha=0.9, edgecolor='none', boxstyle='round,pad=0.3'))

plt.suptitle("Distribuição de tipos de equipamentos x falhas", 
             fontsize=16, fontweight='bold', color=cor_titulo, x=0.08, y=0.98, ha='left')

ax.legend(frameon=False, loc='upper right', bbox_to_anchor=(1, 1.05), ncol=2, fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
freq_falhas = df['falha_maquina'].value_counts()
freq_total = freq_falhas.sum()
perc_falhas = freq_falhas / freq_total*100
falhas = pd.DataFrame({'freq_abs':freq_falhas, 'freq_perc':perc_falhas})
display(falhas)

Equipamentos tipo 'Low' apresentam a maior taxa de falha proporcional

Apesar do tipo Medium e High possuírem volumes diferentes, a taxa de quebra se mantém próxima de 3%

Também fica evidente o desbalanceamento entre as classes em `falha_maquina`, com apenas 3,39% das ocorrências

In [ ]:
'''
Construção de um gráfico matriz de correlações entre as variáveis numéricas contínuas
'''
sns.set_theme(style="white")
plt.rcParams['font.family'] = 'sans-serif'

cor_texto = 'dimgrey'
cor_titulo = 'darkslategray'

# cálculo da matriz de correlação (Pearson)
matriz_corr = df[cols_num_continuas].corr()

rotulos_corr = [
    'Temp. Ar', 'Temp. Processo', 'Velocidade (RPM)', 'Torque', 'Desgaste Ferramenta']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    matriz_corr, 
    cmap='coolwarm', 
    vmax=1, vmin=-1, center=0,
    annot=True, fmt=".2f", 
    annot_kws={"size": 10, "color": cor_texto},
    square=True, linewidths=.5,
)

ax.set_xticklabels(rotulos_corr, rotation=45, ha='right', color=cor_texto, fontsize=10)
ax.set_yticklabels(rotulos_corr, rotation=0, color=cor_texto, fontsize=10)
plt.suptitle("Matriz de correlações lineares", 
             fontsize=16, fontweight='bold', color=cor_titulo, x=0.05, y=0.98, ha='left')

plt.show()


No heatmap é possível apenas enxergar correlações entre variáveis contínuas

A matriz de correlações mostra uma forte correlação linear negativa entre Torque e Velocidade. Como nenhuma delas é variável alvo, devemos verificar se esta relação é estatística e significativa. Neste caso, estas duas grandezas são relacionadas pelas próprias leis da física (velocidade = torque x velocidade). Sendo assim, devemos tratar com cuidado estas variáveis para evitar multicolinearidade num modelo preditivo.

A variável alvo é categórica, portanto não detectaremos aqui correlações com esta variável. Para isto, usaremos outra estratégia , de faremos box plots estratificados.

Para entender se e como as variáveis contínuas tem relação com o comportamento de falha, a estratégia será visualizar as densidades de forma separada, sobreposta e normalizada para cada grupo de ocorrências (falha ou normalidade). O tipo de gráfico adotado será o KDE (Kernel Density Estimate), que é similar a um histograma, mas em forma de curva, e que permite sobrepor duas curvas de cada grupo.

In [ ]:
'''
Construção de uma matriz de gráficos KDE para as variáveis numéricas contínuas.
Cada gráfico será composto de uma curva de distribuição normalizada, equivalente
a cada um dos grupos de ocorrências (normal ou falha)
'''

sns.set_theme(style="white")
plt.rcParams['font.family'] = 'sans-serif'

cor_normal = 'slategrey'
cor_falha = 'crimson'
cor_texto = 'dimgrey'
cor_titulo = 'darkslategray'

lista_rotulos = ['Temperatura do Ar (K)', 'Temperatura do Processo (K)',
                 'Velocidade de Rotação (RPM)', 'Torque (Nm)', 'Desgaste da Ferramenta (min)']
rotulos = dict(zip(cols_num_continuas, rotulos))

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 9))
axs_flat = axs.flat

for i, v in enumerate(cols_num_continuas):
    ax = axs_flat[i]
    
    # Gráfico de densidade sobreposto por classe de falha
    sns.kdeplot(
        data=df, x=v, hue='falha_maquina', ax=ax,
        palette={0: cor_normal, 1: cor_falha},
        fill=True, alpha=0.3, linewidth=2, common_norm=False
    )
    
    ax.set_title(rotulos[v], fontsize=12, fontweight='bold', color=cor_titulo, pad=12)
    ax.set_xlabel('')  # Remove o rótulo do eixo X pois o título já descreve a variável
    ax.set_ylabel('Densidade', fontsize=10, color=cor_texto)
    ax.tick_params(axis='both', colors=cor_texto, labelsize=10)
    
    # Remove a legenda padrão gerada pelo Seaborn para evitar redundância em cada subgráfico
    if ax.get_legend():
        ax.get_legend().remove()
        
    sns.despine(ax=ax)

axs_flat[-1].set_visible(False)

fig.legend(
    labels=['Falha (1)', 'Normal (0)'], loc='upper right', 
    bbox_to_anchor=(0.95, 0.96), ncol=2, frameon=False,
    fontsize=11,handles=[
        plt.Line2D([0], [0], color=cor_falha, lw=4),
        plt.Line2D([0], [0], color=cor_normal, lw=4)
    ])

plt.suptitle("Distribuição de Densidade: Falha x Normal", 
             fontsize=18, fontweight='bold', color=cor_titulo, x=0.05, y=0.98, ha='left')

plt.subplots_adjust(hspace=0.4, wspace=0.3, top=0.82, bottom=0.08, left=0.05, right=0.95)
plt.show()


Podemos observar diferenças entre os padrões de distribuição geral exibidas no 1o gráfico e nos gráficos estratificados. Curvas distantes umas das outras podem indicar indicar relevância da variável para o modelo poder separação classes.

Curvas muito distantes (como Torque e Velocidade) indicam forte poder de separação para o modelo.\nCurvas muito sobrepostas (como as Temperaturas) exigirão combinações de variáveis

### Fase 2: Limpeza e Tratamento de Dados (Data Prep)

- Identificação de duplicados

In [ ]:
duplicados_id = df.duplicated(subset = ['udi']).sum()
duplicados_geral = df.duplicated(
            subset =['udi','id_produto','tipo', 'temperatura_ar_k','temperatura_processo_k',
            'velocidade_rotacao_rpm','torque_nm','desgaste_ferramenta_min', 'falha_maquina']
            ).sum()
duplicados_sem_id = df.duplicated(
            subset =['tipo', 'temperatura_ar_k','temperatura_processo_k',
            'velocidade_rotacao_rpm','torque_nm','desgaste_ferramenta_min', 'falha_maquina']
            ).sum()

print (f'Entradas com identificadores (udi) duplicados.: {duplicados_id}')
print (f'Entradas duplicadas em geral..................: {duplicados_geral}')
print (f'Entradas duplicadas desconsiderando id_produto: {duplicados_sem_id}')

Não foram identificadas entradas com identificadores duplicados (entradas registradas erroneamente por falha sistêmica).\
Também não foram encontradas duplicatas no conjunto geral de dados sendo analisados.\
Foram encontradas 185 duplicatas exatas quando considerados somente os dados dos sensores, igorando o identificador da máquina `id_produto`.\
Estas duplicatas serão identificadas para uma análise mais detalhada.

In [ ]:
idx_duplicadas = df[df.duplicated(subset =['tipo', 'temperatura_ar_k','temperatura_processo_k', 'velocidade_rotacao_rpm',
               'torque_nm','desgaste_ferramenta_min', 'falha_maquina']) == True].index
df_duplicado = df.iloc[idx_duplicadas,:]
display(df_duplicado.sample(10))

In [ ]:
print('\nQuantidade de nulos no dataset de duplicados:\n')
print(df_duplicado.isnull().sum())

As 185 entradas nulas são no mesmo número de entradas duplicadas. Concluímos que foram marcadas como duplicatas por serem todas nulas, não por serem de fato duplicadas. Elas não serão removidas e serão tratadas na etapa seguinte, que trata das entradas ausentes.

- Tratamento de dados ausentes

In [ ]:
print('\nQuantidade de nulos no dataset geral:\n')
print(df.isnull().sum())

Analisando as estatísticas descritivas e os histogramas, observamos que `desgaste_ferramentas_min` e `velocidade_rotacao_rpm` apresentam distribuições distorcidas em relação a uma normal e portanto não poderemos adotar a média como parâmetro para imputação de dados ausentes. Como média e mediana são muito próximas em todas as outras variáveis, será utilizada a mediana como critério para imputação em todas as colunas. 

- Imputação de dados ausentes por mediana

In [ ]:
df_limpo = df.fillna(
    {'temperatura_ar_k' : df['temperatura_ar_k'].median(),
     'temperatura_processo_k': df['temperatura_processo_k'].median(),
     'velocidade_rotacao_rpm': df['velocidade_rotacao_rpm'].median(),
     'torque_nm' : df['torque_nm'].median()}
)

In [ ]:
print('\nQuantidade de nulos no dataset tratado:\n')
df_limpo.isnull().sum()

- Identificação de Ouliers

In [ ]:
'''
Construção de gráficos tipo boxplot para variáveis numéricas contínuas
para identificação de possíveis outliers
'''
sns.set_theme(style="white")
plt.rcParams['font.family'] = 'sans-serif'

cor_normal = 'slategrey'
cor_texto = 'dimgrey'
cor_titulo = 'darkslategray'

lista_rotulos = ['Temperatura do Ar (K)', 'Temperatura do Processo (K)',
                 'Velocidade de Rotação (RPM)', 'Torque (Nm)', 'Desgaste da Ferramenta (min)']
rotulos = dict(zip(cols_num_continuas, rotulos))

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 9))
axs_flat = axs.flat

for i, v in enumerate(cols_num_continuas):
    ax = axs_flat[i]
    
    # Boxplot comparando a distribuição do sensor para Máquina Normal (0) vs Falha (1)
    sns.boxplot(data=df[v], ax=ax, width=0.4, fliersize=3, legend=False)

    ax.set_title(rotulos[v], fontsize=12, fontweight='bold', color=cor_titulo, pad=10)
    ax.set_ylabel('') # Remove o rótulo do eixo Y pois o título já descreve o sensor
    ax.set_xlabel('') # Remove o rótulo padrão do eixo X
    ax.tick_params(axis='y', colors=cor_texto, labelsize=10)
    sns.despine(ax=ax)

axs_flat[-1].set_visible(False)

plt.suptitle("Distribuição de frequências com boxplots: Presença de Outliers em Torque e Velocidade", 
             fontsize=18, fontweight='bold', color=cor_titulo, x=0.05, y=0.98, ha='left')

plt.subplots_adjust(hspace=0.4, wspace=0.3, top=0.84, bottom=0.08, left=0.05, right=0.95)
plt.show()


Foram detectados outliers nas colunas `velocidade_rotacao_rpm` e `torque_nm`, o que faz sentido dada a relação entre as duas grandezas já mencionada anteriormente.

Em princípio, não é possível saber se são entradas inconsistentes. Elas serão identificadas em um Dataframe à parte, para eventuais consultas pela equipe de engenharia.

In [ ]:
'''
Função que recebe dataframe e coluna a ser analisada,
verifica os valores que estão fora dos limites e retorna os seus respectivos índices
'''
def detecta_outlier (df, coluna):
    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3-q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    outliers = df[coluna].map(lambda x: 1 if (x <= limite_inf or x >= limite_sup) else 0)
    idx_outliers = outliers[outliers == 1].index  
      
    return idx_outliers

In [ ]:
# identifica outliers na coluna de velocidade
idx_out_velocidade = detecta_outlier(df_limpo, 'velocidade_rotacao_rpm') 
# identifica outliers na coluna de torque
idx_out_torque = detecta_outlier(df_limpo, 'torque_nm')
# gera um indice geral sem repetição através de conjunto (caso a condição de outlier esteja presente em ambas as colunas)
unifica_outliers = set(list(idx_out_velocidade) + list(idx_out_torque))
# gera Dataframe só de outliers
idx_outliers = list(unifica_outliers)
print (f'Detectadas {len(idx_outliers)} linhas com valores discrepantes (outliers)')

### Fase 3: Feature Engineering

In [ ]:
df_limpo.columns

- Feature `potencia` derivada de `velocidade_rotacao_rpm` e `torque_nm`. Potência é uma grandeza da mecânica que é derivada da velocidade e potência. Como visto na anális exploratória, a correlação entre ambas caracteriza multicolinearidade, que é prejudicial aos modelos. Uma única feature que represente este comportamento ajuda a melhorar o desempenho da análise.

In [ ]:
df_limpo['potencia'] = df_limpo['velocidade_rotacao_rpm'] * df_limpo['torque_nm']

- Feature `ganho_temperatura`: estabelece uma razão entre a temperatura gerada no processo `temperatura_processo_k` e a temperatura ambiente `temperatura_ar_k`. A ideia é que o valor absoluto das temperaturas analisados isoladamente não necessariamente predizem a falha, mas o quanto a temperatura variou em termos proporcionais. Com isso, também reduzimos a dimensionalidade e possibilidade de overfitting

In [ ]:
df_limpo['ganho_temperatura'] = df_limpo['temperatura_processo_k'] / df_limpo['temperatura_ar_k']

### Fase 4: Divisão e Balanceamento dos Dados

- Separação das variáveis preditoras e variável alvo

In [ ]:
colunas_manter = ['tipo','desgaste_ferramenta_min','potencia', 'ganho_temperatura']
X = df_limpo[colunas_manter]
y = df_limpo['falha_maquina']

- Divisão dos dados em treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=42)

- Reamostragem de dados de treino

A classe `falha (1)` é esmagadoramente menor que a classe `normal (0)`. A reamostragem por undersampling neste caso ocasionará perda considerável de dados, já que ela reduz as ocorências aleatoriamente da classe majoritária para equalizar com a minoritária.

Vamos aplicar o SMOTE, técnica de oversampling, ou seja, infla sinteticamente a classe minoritária para equilibrar as ocorrências. Como baseia-se num algoritmo parecido como KNN para criar estes novos dados, faz sentido utilizá-la no modelo KNN. Na árvore de decisão usaremos o próprio parãmetro do modelo para tratar este desbalanceamento (`class_weight = balanced`)

### Fase 5: Escalonamento de Variáveis

O modelo KNN é um modelo de vizinhança, que elege a classe à qual pertence a entidade baseado na semelhança com `n` vizinhos mais próximos. Esta proximidade é calculada pela distância euclidiana e a diferença de magnitude das grandezas expressas nas variáveis preditoras farão com que as de maior valor absoluto sobressaiam em termos de importância.

Por este motivo é necessário aplicar a padronização nos dados quantitativos que alimentarão este modelo. Esta padronização será feita antes do oversampling.

In [ ]:
scaler = StandardScaler()
colunas_continuas = ['desgaste_ferramenta_min', 'potencia', 'ganho_temperatura']

X_train_escalonado = scaler.fit_transform(X_train[colunas_continuas])
X_test_escalonado = scaler.transform(X_test[colunas_continuas])

smote = SMOTE(random_state=42)

X_train_knn, y_train_knn = smote.fit_resample(X_train_escalonado, y_train)

print(f"Antes do SMOTE (Treino): {y_train.value_counts().to_dict()}")
print(f"Após o SMOTE (Treino): {y_train_knn.value_counts().to_dict()}")


Para a Árvore de Decisão, o escalonamento não faz diferença pois é um modelo que busca separar o conjunto de dados através de linhas de corte bem definidas, considerando individualmente cada variável. Assim, as diferenças de escalas entre elas não mudam o resultado final.
A diferença é que será mantida a coluna categórica `tipo`.